# Gộp output arXiv crawler: CPU part 1 + 2 + 3

Notebook chạy **trên Kaggle**, dùng CPU, không cần Internet, token hoặc cài thêm thư viện.
Chỉ đọc các output đã gắn vào `/kaggle/input`; không crawl lại hoặc tự upload lên Hugging Face.

## Cách chạy

1. Import file `.ipynb` này vào một notebook Kaggle mới.
2. Trong **Add Input / Add Data**, tìm **Notebook Output** và gắn output đã lưu của cả ba notebook private bên dưới bằng tài khoản có quyền đọc. Nếu ô tìm kiếm không nhận URL, tìm theo tên notebook.
3. Chọn CPU, tắt Internet. Chạy các cell theo thứ tự. Cell nhận diện input phải in đúng ba thư mục `arxiv-part1`, `arxiv-part2`, `arxiv-part3`. Nếu đường dẫn mount khác, điền `SOURCE_ROOTS`.
4. **Save Version → Save & Run All**, chờ hoàn tất rồi tải `arxiv-merged.zip` trong tab **Output** (hoặc link ở cell cuối).

Nguồn đã đối chiếu từ giao diện notebook ngày 2026-11-06:

| Part | Notebook | Lát cắt dòng ID (0-based, đầu đóng/cuối mở) |
|---|---|---|
| 1 | [part 1](https://www.kaggle.com/code/nhminh107/arxiv-crawler-cpu-part-1-20261024) | `[0, 2829)` |
| 2 | [part 2](https://www.kaggle.com/code/nhminh107/arxiv-crawler-cpu-part-2-20261024) | `[2829, 5657)` |
| 3 | [part 3](https://www.kaggle.com/code/nhminh107/arxiv-crawler-cpu-part-3-20261024) | `[5657, 8486)` |

Đây là **8.486 Version ID dự kiến**, không phải số Publication hay bằng chứng crawl đủ.
Notebook kiểm tra lại danh sách ID thật trong input, không áp dụng Range `2508` trong cấu hình repo.

## Quy tắc gộp

- Giữ nguyên byte của archive `.tar.gz`, source `.tex`/`.bib`, `status.json` và các file khác trong `output/`.
- Cùng đường dẫn chỉ giữ một bản khi **SHA-256 giống nhau**; khác nội dung thì ghi `merge-conflicts.json` và dừng, không tự chọn bản thắng.
- `metadata.json` vẫn là log tải với các trường gốc; giữ mọi lần thử, chỉ đổi `path` sang đường dẫn tương đối `output/...` để dùng được sau khi giải nén. Bản gốc của từng part nằm trong `provenance/`.
- Tính lại `done.json` từ toàn bộ danh sách ID và `status.json` đã gộp, kể cả Publication có Version nằm ở hai part. Giữ bản `done.json`/`pass.txt` cũ trong `provenance/`.
- Version `empty` vẫn được giữ. Version `error`, `downloaded`, thiếu status/archive/file đã khai báo được ghi rõ trong báo cáo; không tự sửa hoặc tải bù.
- ZIP được ghi theo luồng trực tiếp từ input, không tạo thêm một bản sao đầy đủ của cây dữ liệu trên ổ đĩa. RAM chủ yếu chứa chỉ mục tên file, metadata và các JSON nhỏ; nội dung file lớn được đọc từng khối.

Output này là **bản sao lưu corpus thô**, chưa phải sáu bảng Parquet nộp cuối kỳ. `metadata.json` của crawler không phải bảng `papers`.


In [ ]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
SOURCES = {
    1: "nhminh107/arxiv-crawler-cpu-part-1-20261024",
    2: "nhminh107/arxiv-crawler-cpu-part-2-20261024",
    3: "nhminh107/arxiv-crawler-cpu-part-3-20261024",
}

# Để trống để tự tìm. Nếu cần, điền đúng thư mục chứa metadata.json và output/.
SOURCE_ROOTS = {
    # 1: Path("/kaggle/input/.../arxiv-part1"),
    # 2: Path("/kaggle/input/.../arxiv-part2"),
    # 3: Path("/kaggle/input/.../arxiv-part3"),
}
EXPECTED_VERSION_COUNT = 8486
ZIP_PATH = WORK_DIR / "arxiv-merged.zip"
VERIFY_ZIP_CRC = True  # Đọc lại ZIP để kiểm tra CRC; tốn thêm thời gian I/O.


In [ ]:
import csv
import hashlib
import io
import os
import json
import re
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, indent=2) + "\n").encode("utf-8")


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def version_location(version_id):
    if not re.fullmatch(r"(?:\d{4}\.\d{4,5}|[a-zA-Z][a-zA-Z0-9.-]*/\d{7})v[1-9]\d*", version_id):
        raise ValueError(f"Version ID không hợp lệ: {version_id!r}")
    base_id, version = version_id.rsplit("v", 1)
    key = base_id.replace("/", "-").replace(".", "-")
    return base_id, int(version), f"output/{key}/v{int(version)}"


def discover_sources(input_dir, overrides):
    candidates = []
    # Dừng ngay ở crawl root; không liệt kê cả cây source trên Kaggle mount.
    for directory, children, names in os.walk(input_dir):
        if "metadata.json" in names and "output" in children:
            candidates.append(Path(directory))
            children.clear()
        elif "output" in children:
            children.remove("output")
    roots = {}
    for part in SOURCES:
        matches = ([Path(overrides[part])] if part in overrides else
                   [p for p in candidates if p.name == f"arxiv-part{part}"])
        if len(matches) != 1:
            raise ValueError(f"Part {part}: tìm thấy {len(matches)} nguồn. "
                             f"Gắn đúng 3 Notebook Output hoặc điền SOURCE_ROOTS. Ứng viên: {candidates}")
        root = matches[0].resolve()
        if not (root / "metadata.json").is_file() or not (root / "output").is_dir():
            raise ValueError(f"Nguồn không có metadata.json và output/: {root}")
        roots[part] = root
    if len(set(roots.values())) != len(SOURCES):
        raise ValueError("Các part đang trỏ tới cùng một thư mục.")
    return roots


def build_merge_plan(roots, expected_count, report_dir):
    """Chỉ mục file và kiểm tra độ phủ trước khi ghi ZIP; không sửa input."""
    files, fingerprints, generated = {}, {}, {}
    conflicts, duplicates, metadata, expected = [], [], [], set()
    source_summary = []

    def add_file(relative, source):
        size = source.stat().st_size
        digest = sha256_file(source)
        if relative in files:
            event = {"path": relative, "first": str(files[relative]), "second": str(source)}
            if fingerprints[relative] != (size, digest):
                conflicts.append(event)
            else:
                duplicates.append(event)
            return
        files[relative] = source
        fingerprints[relative] = (size, digest)

    for part, root in sorted(roots.items()):
        # Crawler ghi toàn bộ danh sách ID ở cùng cấp arxiv-partN/.
        ids_path = root.parent / "arxiv_id.txt"
        if not ids_path.is_file():
            raise ValueError(f"Thiếu {ids_path}; cần gắn toàn bộ Notebook Output.")
        ids = {line.strip() for line in ids_path.read_text(encoding="utf-8").splitlines() if line.strip()}
        for version_id in ids:
            version_location(version_id)
        if expected and expected != ids:
            raise ValueError(f"Danh sách arxiv_id.txt của part {part} khác các part trước.")
        expected = ids
        add_file(f"provenance/part{part}/arxiv_id.txt", ids_path)
        add_file(f"provenance/part{part}/metadata.json", root / "metadata.json")
        rows = read_json(root / "metadata.json")
        if not isinstance(rows, list):
            raise ValueError(f"metadata.json phải là list: {root}")
        for row in rows:
            version_location(row["id"])
            # Giữ thứ tự lịch sử retry trong mỗi part, không chọn success/fail thay người dùng.
            metadata.append({**row, "path": f"output/{row['id']}.tar.gz"})
        count = 0
        for source in sorted((root / "output").rglob("*")):
            if source.is_symlink():
                raise ValueError(f"Không gộp symlink: {source}")
            if not source.is_file():
                continue
            relative = source.relative_to(root).as_posix()
            parts = Path(relative).parts
            # Chỉ xử lý control file đúng vị trí, không nhầm file LaTeX phụ cùng tên.
            is_done = len(parts) == 3 and parts[-1] == "done.json"
            if is_done or relative == "output/pass.txt":
                relative = f"provenance/part{part}/{relative}"
            add_file(relative, source)
            count += 1
            if count % 2000 == 0:
                print(f"Part {part}: đã kiểm tra {count:,} file", flush=True)
        source_summary.append({"part": part, "notebook": SOURCES[part], "root": str(root),
                               "files": count, "metadata_rows": len(rows),
                               "metadata_versions": len({r["id"] for r in rows})})

    report_dir.mkdir(parents=True, exist_ok=True)
    if conflicts:
        (report_dir / "merge-conflicts.json").write_bytes(json_bytes(conflicts))
        raise ValueError(f"Có {len(conflicts)} đường dẫn khác nội dung; xem merge-conflicts.json. Chưa tạo ZIP.")
    if expected_count is not None and len(expected) != expected_count:
        raise ValueError(f"Có {len(expected)} ID dự kiến, cấu hình yêu cầu {expected_count}.")
    if not expected:
        raise ValueError("Danh sách ID dự kiến rỗng.")

    observed = {r["id"] for r in metadata}
    groups = defaultdict(list)
    version_report = []
    for version_id in sorted(expected | observed):
        base_id, version, directory = version_location(version_id)
        status_path = f"{directory}/status.json"
        status = read_json(files[status_path]) if status_path in files else {}
        missing = []
        if not status:
            missing.append("status.json")
        elif status.get("version") != version:
            missing.append("status.version không khớp ID")
        declared = status.get("files", [])
        if status.get("status") == "ok" and not declared:
            missing.append("status=ok nhưng files rỗng")
        for name in declared:
            path = Path(name)
            if path.is_absolute() or ".." in path.parts:
                missing.append(f"Đường dẫn không hợp lệ trong status.files: {name}")
            elif f"{directory}/{path.as_posix()}" not in files:
                missing.append(name)
        archive_present = f"output/{version_id}.tar.gz" in files
        if status.get("status") in {"ok", "empty", "downloaded"} and not archive_present:
            missing.append("archive .tar.gz")
        complete = status.get("status") in {"ok", "empty"} and not missing
        entry = {"id": version_id, "expected": version_id in expected,
                 "status": status.get("status", "missing"), "archive_present": archive_present,
                 "complete": complete, "issues": missing}
        version_report.append(entry)
        groups[base_id].append((version_id, status, entry))

    done_count = 0
    for base_id, records in groups.items():
        # Chỉ công nhận done khi đủ mọi version của Publication trong danh sách gốc.
        if not all(entry["complete"] for _, _, entry in records):
            continue
        key = base_id.replace("/", "-").replace(".", "-")
        done = {"id": base_id, "status": "done", "n_versions": len(records),
                "versions_with_tex": sum(s["status"] == "ok" for _, s, _ in records)}
        for field in ("bytes_downloaded", "bytes_before", "bytes_after", "seconds"):
            done[field] = round(sum(s.get(field, 0) for _, s, _ in records), 3)
        generated[f"output/{key}/done.json"] = json_bytes(done)
        done_count += 1

    incomplete = [e["id"] for e in version_report if not e["complete"]]
    no_tex = [e["id"] for e in version_report if e["status"] == "empty"]
    # pass.txt của crawler chứa cả empty và lỗi giải nén; giữ ý nghĩa đó và bổ sung missing.
    generated["output/pass.txt"] = ("".join(f"{v}\n" for v in sorted(set(incomplete + no_tex)))).encode()
    generated["arxiv_id.txt"] = ("".join(f"{v}\n" for v in sorted(expected))).encode()
    generated["metadata.json"] = json_bytes(metadata)
    generated["reports/versions.json"] = json_bytes(version_report)
    generated["reports/duplicates.json"] = json_bytes(duplicates)
    generated["reports/incomplete_ids.txt"] = ("".join(f"{v}\n" for v in incomplete)).encode()
    report = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "sources": source_summary, "expected_versions": len(expected),
        "expected_publications": len({version_location(v)[0] for v in expected}),
        "observed_metadata_versions": len(observed), "metadata_rows_including_retries": len(metadata),
        "expected_ids_without_metadata": sorted(expected - observed),
        "unexpected_metadata_ids": sorted(observed - expected),
        "version_status_counts": dict(Counter(e["status"] for e in version_report)),
        "incomplete_versions": len(incomplete), "done_publications": done_count,
        "identical_duplicate_files": len(duplicates), "conflicting_files": 0,
        "crawl_complete_for_expected_ids": not incomplete and not (expected - observed) and not (observed - expected),
    }
    generated["reports/merge_report.json"] = json_bytes(report)
    generated["README.md"] = (
        "# Merged arXiv crawl backup\n\n"
        "Sources: CPU parts 1, 2, 3 (20261024).\n"
        "output/: unchanged crawl archives, extracted sources and per-version status.\n"
        "done.json and pass.txt were rebuilt after merging. Original control files are in provenance/.\n"
        "metadata.json retains all download attempts; paths are relative to this folder.\n"
        "See reports/merge_report.json and reports/versions.json for completeness and issues.\n"
        "checksums.csv lists SHA-256 for every payload file except checksums.csv itself.\n"
        "This is a raw backup, not the final six-table Parquet dataset.\n"
    ).encode()
    return files, fingerprints, generated, report


def write_merge_zip(zip_path, files, fingerprints, generated, verify_crc=True):
    """Ghi ZIP64 trực tiếp, giữ archive đã nén ở chế độ STORE, kiểm tra byte khi ghi."""
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    partial = zip_path.with_suffix(".zip.partial")
    if zip_path.exists() or partial.exists():
        raise FileExistsError(f"Output đã tồn tại; đổi ZIP_PATH để chạy lại an toàn: {zip_path}")
    required = sum(size for size, _ in fingerprints.values()) + sum(map(len, generated.values()))
    reserve = max(64 * 1024**2, required // 50)
    free = shutil.disk_usage(zip_path.parent).free
    print(f"Dữ liệu trước nén: {required / 1024**3:.2f} GiB; ổ đĩa trống: {free / 1024**3:.2f} GiB")
    if free < required + reserve:
        raise OSError("Không đủ dung lượng dự phòng để ghi ZIP chứa toàn bộ archive và source.")
    if set(files) & set(generated):
        raise ValueError("File sinh ra trùng đường dẫn input.")
    checksums = io.StringIO()
    writer = csv.writer(checksums)
    writer.writerow(["path", "bytes", "sha256"])
    with zipfile.ZipFile(partial, "x", compression=zipfile.ZIP_DEFLATED,
                         compresslevel=1, allowZip64=True) as archive:
        for index, (relative, source) in enumerate(sorted(files.items()), 1):
            name = f"arxiv-merged/{relative}"
            info = zipfile.ZipInfo(name)
            info.compress_type = (zipfile.ZIP_STORED if relative.endswith((".gz", ".zip"))
                                  else zipfile.ZIP_DEFLATED)
            digest, size = hashlib.sha256(), 0
            with source.open("rb") as incoming, archive.open(info, "w", force_zip64=True) as outgoing:
                for chunk in iter(lambda: incoming.read(8 * 1024 * 1024), b""):
                    outgoing.write(chunk)
                    digest.update(chunk)
                    size += len(chunk)
            if (size, digest.hexdigest()) != fingerprints[relative]:
                raise ValueError(f"File thay đổi trong lúc gộp: {source}. ZIP chưa được công nhận hoàn tất.")
            writer.writerow([relative, size, digest.hexdigest()])
            if index % 2000 == 0:
                print(f"Đã đóng gói {index:,}/{len(files):,} file", flush=True)
        for relative, content in sorted(generated.items()):
            archive.writestr(f"arxiv-merged/{relative}", content)
            writer.writerow([relative, len(content), hashlib.sha256(content).hexdigest()])
        archive.writestr("arxiv-merged/checksums.csv", checksums.getvalue())
    with zipfile.ZipFile(partial) as archive:
        if len(archive.namelist()) != len(files) + len(generated) + 1:
            raise ValueError("Số file trong ZIP không khớp kế hoạch.")
        if verify_crc:
            print("Đang đọc lại ZIP để kiểm tra CRC...", flush=True)
            bad = archive.testzip()
            if bad is not None:
                raise ValueError(f"ZIP CRC lỗi: {bad}")
    partial.rename(zip_path)
    checksum_path = zip_path.with_suffix(".zip.sha256")
    checksum_path.write_text(f"{sha256_file(zip_path)}  {zip_path.name}\n", encoding="utf-8")
    return checksum_path


## 1. Nhận diện ba nguồn input

In [ ]:
print("Đang nhận diện thư mục gốc của 3 input...", flush=True)
roots = discover_sources(INPUT_DIR, SOURCE_ROOTS)
for part, root in roots.items():
    print(f"Part {part}: {root}", flush=True)
    print("  File/thư mục:", ", ".join(p.name for p in sorted(root.iterdir())))


## 2. Kiểm tra file, xung đột và độ phủ Version
Đọc SHA-256 từng file; chưa ghi ZIP.

In [ ]:
started = time.perf_counter()
files, fingerprints, generated, report = build_merge_plan(
    roots, EXPECTED_VERSION_COUNT, WORK_DIR
)
(WORK_DIR / "merge_report.json").write_bytes(json_bytes(report))
print(json.dumps(report, ensure_ascii=False, indent=2))
if not report["crawl_complete_for_expected_ids"]:
    print("LƯU Ý: gộp vẫn giữ toàn bộ file hiện có, nhưng crawl chưa đủ hoặc có bất nhất. "
          "Xem reports/versions.json và reports/incomplete_ids.txt trong ZIP; không có tải bù.")


## 3. Ghi một ZIP và kiểm tra toàn vẹn
Chỉ sau bước này có link tải ZIP hoàn tất.

In [ ]:
checksum_path = write_merge_zip(
    ZIP_PATH, files, fingerprints, generated, verify_crc=VERIFY_ZIP_CRC
)
print(f"ZIP hoàn tất: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024**3:.2f} GiB)")
print(f"Thời gian gộp + kiểm tra: {time.perf_counter() - started:.1f} giây")
from IPython.display import FileLink, display
display(FileLink(str(ZIP_PATH)))
display(FileLink(str(checksum_path)))
display(FileLink(str(WORK_DIR / "merge_report.json")))


## Sau khi tải về

Tải `arxiv-merged.zip`, `arxiv-merged.zip.sha256` và đọc `merge_report.json` trước khi sử dụng.
Giải nén ZIP sẽ có:

```text
arxiv-merged/
  arxiv_id.txt
  metadata.json                 # mọi lần thử tải; path tương đối với thư mục này
  output/
    <version-id>.tar.gz         # archive gốc, có thể chứa hình như lúc crawl
    <publication-key>/
      v1/status.json
      v1/...tex, ...bib
      v2/...
      done.json                 # tính lại từ mọi Version trong danh sách ID
    pass.txt                    # empty + chưa hoàn tất; không đồng nghĩa tất cả đều lỗi
  provenance/part1/ ...part2/ ...part3/
  reports/merge_report.json
  reports/versions.json
  reports/incomplete_ids.txt
  reports/duplicates.json
  checksums.csv
  README.md
```

Bạn có thể upload ZIP làm **bản sao lưu** vào dataset Hugging Face private của mình.
Để Parser đọc từng file trực tiếp theo quy ước repo, giải nén và đưa **các thư mục Publication** từ `output/` vào `raw/<member>/`, giữ nguyên `<publication-key>/v<n>/`;
archive gốc và báo cáo có thể lưu ở khu vực backup riêng. Chọn `<member>` đúng thành viên sở hữu corpus này.
Notebook không tạo bảng `papers`/`references`, không đổi schema Parquet và không upload.

Nếu gặp lỗi:

- **Không tìm thấy input:** gắn output của cả ba notebook rồi xem đường dẫn trong sidebar và sửa `SOURCE_ROOTS`. Đừng chỉ gắn file `.ipynb`.
- **Xung đột:** đọc `merge-conflicts.json`, kiểm tra lại version output đã chọn. Notebook không ghi đè file khác nội dung.
- **Thiếu dữ liệu:** `reports/versions.json` chỉ rõ Version và file thiếu. `crawl_complete_for_expected_ids=false` không ngăn tạo bản sao lưu nhưng không chứng nhận crawl đủ.
- **Thiếu dung lượng:** notebook dừng trước khi đóng gói. ZIP chứa cả archive gốc nên dung lượng có thể lớn; không tự xóa file để tiếp tục.
- **Chạy lại:** nếu ZIP hoặc `.zip.partial` đã có, đổi `ZIP_PATH` hoặc chạy phiên Kaggle sạch. Notebook không xóa hay ghi đè output cũ; không resume ZIP dở.

Tham khảo: [ba nguồn ở đầu notebook](https://www.kaggle.com/code/nhminh107/arxiv-crawler-cpu-part-1-20261024),
[Kaggle: Notebook Output / kernel sources](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels_metadata.md),
[đề Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).

**Kiểm thử trước bàn giao:** chạy bằng dữ liệu tự tạo trong test, không tải corpus thật về máy phát triển. Lượt gộp toàn bộ 8.486 Version chỉ được xác minh sau khi bạn chạy notebook trên Kaggle.
